# 02 — Annual FTR Residual Analysis

**Goal:** Reconstruct baselines, compute residuals, compare to f0p.

Resolves: Q1 (per-quarter difficulty), Q3 (band widths), A1 (narrower bands?), A4 (R2/R3 feasibility), A5 (aq4 structural difficulty)

In [ ]:
import os, warnings
warnings.filterwarnings('ignore')
os.chdir('/home/xyz/workspace/pmodel/src')

from pbase.config.ray import init_ray
import pmodel
init_ray(extra_modules=[pmodel])

from pbase.analysis.utils import misc
misc.logging_setup()

import pandas as pd
import numpy as np
import glob

pd.options.display.float_format = '{:.2f}'.format
pd.set_option('display.max_columns', 99)
pd.set_option('display.max_rows', 200)

from pbase.analysis.tools.all_positions import MisoApTools
aptools = MisoApTools()
miso_tools = aptools.tools
print('Initialized')

## 2.1 Baseline Reconstruction (R1: fill_mtm_1st_period_with_hist_revenue)

In [ ]:
# Load annual cleared trades
annual_all = pd.read_parquet('/opt/temp/qianli/annual_research/annual_cleared_all.parquet')
print(f'Annual rows: {len(annual_all)}')
print(f'Period types: {annual_all["period_type"].value_counts().to_dict()}')
print(f'Rounds: {annual_all["round"].value_counts().sort_index().to_dict()}')

# For R1 trades: reconstruct H via fill_mtm_1st_period_with_hist_revenue
# This function needs trades grouped by (auction_date, period_type, class_type)
r1_trades = annual_all[annual_all['round'] == 1].copy()
print(f'\nR1 trades: {len(r1_trades)}')

r1_with_h = []
groups = r1_trades.groupby(['planning_year', 'period_type', 'class_type'])
for (py, ptype, ctype), gdf in groups:
    print(f'  Filling H for PY={py}, {ptype}, {ctype} ({len(gdf)} rows)...')
    try:
        filled = miso_tools.fill_mtm_1st_period_with_hist_revenue(gdf)
        r1_with_h.append(filled)
        n_filled = filled['mtm_1st_mean'].notna().sum()
        print(f'    Filled: {n_filled}/{len(gdf)} ({100*n_filled/len(gdf):.1f}%)')
    except Exception as e:
        print(f'    ERROR: {e}')

r1_filled = pd.concat(r1_with_h, ignore_index=True)
print(f'\nR1 filled total: {len(r1_filled)}')
print(f'R1 with H (mtm_1st_mean not NaN): {r1_filled["mtm_1st_mean"].notna().sum()}')

In [ ]:
# R1 residuals: residual = mcp - H
r1_filled['residual'] = r1_filled['mcp'] - r1_filled['mtm_1st_mean']
r1_filled['abs_residual'] = r1_filled['residual'].abs()
r1_filled['baseline'] = r1_filled['mtm_1st_mean']
r1_filled['round_label'] = 'R1'

# Drop rows without baseline
r1_valid = r1_filled[r1_filled['baseline'].notna()].copy()
print(f'R1 valid (with baseline): {len(r1_valid)}')

In [ ]:
# R2/R3 trades: mtm_1st_mean already = previous round's MCP
# But we need to verify this. For cleared trades, the MCP column IS the clearing price.
# For R2/R3, the 'mcp' column is THIS round's MCP. The previous round's MCP is not directly in the data.
# We need to reconstruct: for each R2 trade, find the same path's R1 MCP.

# Strategy: for each (PY, period_type, class_type, path), chain MCPs across rounds
r23_trades = annual_all[annual_all['round'].isin([2, 3])].copy()
print(f'R2/R3 trades: {len(r23_trades)}')

# Build MCP lookup: for each (PY, period_type, class_type, path, round) → mean MCP
mcp_lookup = annual_all.groupby(['planning_year','period_type','class_type','path','round'])['mcp'].mean().reset_index()
mcp_lookup = mcp_lookup.rename(columns={'mcp': 'prev_round_mcp', 'round': 'prev_round'})

# For R2 trades: previous round = R1
r2_trades = r23_trades[r23_trades['round'] == 2].copy()
r2_lookup = mcp_lookup[mcp_lookup['prev_round'] == 1].drop(columns=['prev_round'])
r2_trades = r2_trades.merge(r2_lookup, on=['planning_year','period_type','class_type','path'], how='left')

# For R3 trades: previous round = R2
r3_trades = r23_trades[r23_trades['round'] == 3].copy()
r3_lookup = mcp_lookup[mcp_lookup['prev_round'] == 2].drop(columns=['prev_round'])
r3_trades = r3_trades.merge(r3_lookup, on=['planning_year','period_type','class_type','path'], how='left')

r23_merged = pd.concat([r2_trades, r3_trades], ignore_index=True)
r23_merged['residual'] = r23_merged['mcp'] - r23_merged['prev_round_mcp']
r23_merged['abs_residual'] = r23_merged['residual'].abs()
r23_merged['baseline'] = r23_merged['prev_round_mcp']
r23_merged['round_label'] = 'R' + r23_merged['round'].astype(str)

r23_valid = r23_merged[r23_merged['baseline'].notna()].copy()
print(f'R2 with prev MCP: {len(r2_trades[r2_trades["prev_round_mcp"].notna()])}')
print(f'R3 with prev MCP: {len(r3_trades[r3_trades["prev_round_mcp"].notna()])}')
print(f'R2/R3 valid total: {len(r23_valid)}')

## 2.2 Annual Residual Distributions

In [ ]:
# Combine R1 and R2/R3 residuals
all_residuals = pd.concat([
    r1_valid[['planning_year','round','period_type','class_type','path','mcp','baseline','residual','abs_residual','round_label','cleared_volume']],
    r23_valid[['planning_year','round','period_type','class_type','path','mcp','baseline','residual','abs_residual','round_label','cleared_volume']],
], ignore_index=True)
print(f'All residuals: {len(all_residuals)}')

# Residual stats by round x quarter x class_type
def residual_stats(df, groupby_cols):
    return df.groupby(groupby_cols).agg(
        n=('residual', 'size'),
        mean_residual=('residual', 'mean'),
        median_residual=('residual', 'median'),
        mean_abs_residual=('abs_residual', 'mean'),
        median_abs_residual=('abs_residual', 'median'),
        p25_abs=('abs_residual', lambda x: x.quantile(0.25)),
        p75_abs=('abs_residual', lambda x: x.quantile(0.75)),
        p90_abs=('abs_residual', lambda x: x.quantile(0.90)),
        p95_abs=('abs_residual', lambda x: x.quantile(0.95)),
        p99_abs=('abs_residual', lambda x: x.quantile(0.99)),
    ).reset_index()

# By round x quarter
stats_rq = residual_stats(all_residuals, ['round_label', 'period_type'])
print('\n=== Residual Stats by Round x Quarter ===')
print(stats_rq.to_string(index=False))

In [ ]:
# By round x quarter x class_type
stats_rqc = residual_stats(all_residuals, ['round_label', 'period_type', 'class_type'])
print('=== Residual Stats by Round x Quarter x Class Type ===')
print(stats_rqc.to_string(index=False))

In [ ]:
# Direction accuracy: % where sign(baseline) == sign(mcp)
all_residuals['sign_match'] = (np.sign(all_residuals['mcp']) == np.sign(all_residuals['baseline'])).astype(int)
dir_acc = all_residuals.groupby(['round_label', 'period_type']).agg(
    n=('sign_match', 'size'),
    direction_accuracy=('sign_match', 'mean'),
).reset_index()
print('=== Direction Accuracy (sign match) ===')
print(dir_acc.to_string(index=False))

## 2.3 f0p Residual Comparison

In [ ]:
# Load f0p training data from parquet
import glob

f0p_training_files = glob.glob('/opt/temp/qianli/mcp_pred_training/auction_month=*/period_type=f0/class_type=*/training.parquet')
print(f'Found {len(f0p_training_files)} f0p training files')

f0p_dfs = []
for f in f0p_training_files:
    parts = f.split('/')
    am = [p for p in parts if p.startswith('auction_month=')][0].split('=')[1]
    ct = [p for p in parts if p.startswith('class_type=')][0].split('=')[1]
    df = pd.read_parquet(f)
    df['auction_month'] = am
    df['class_type_dir'] = ct
    f0p_dfs.append(df)

f0p_training = pd.concat(f0p_dfs, ignore_index=True)
print(f'f0p training total: {len(f0p_training)}')
print(f'Columns: {f0p_training.columns.tolist()}')
print(f'Period types: {f0p_training["period_type"].unique().tolist()}')

In [ ]:
# Compute f0p residuals: residual = mcp_mean - mtm_1st_mean (same logic as baseline)
f0p_training['residual'] = f0p_training['mcp_mean'] - f0p_training['mtm_1st_mean']
f0p_training['abs_residual'] = f0p_training['residual'].abs()

f0p_stats = f0p_training.groupby(['class_type']).agg(
    n=('residual', 'size'),
    mean_residual=('residual', 'mean'),
    median_residual=('residual', 'median'),
    mean_abs_residual=('abs_residual', 'mean'),
    median_abs_residual=('abs_residual', 'median'),
    p25_abs=('abs_residual', lambda x: x.quantile(0.25)),
    p75_abs=('abs_residual', lambda x: x.quantile(0.75)),
    p90_abs=('abs_residual', lambda x: x.quantile(0.90)),
    p95_abs=('abs_residual', lambda x: x.quantile(0.95)),
    p99_abs=('abs_residual', lambda x: x.quantile(0.99)),
).reset_index()

print('=== f0p Residual Stats (mcp_mean - mtm_1st_mean) ===')
print(f0p_stats.to_string(index=False))

In [ ]:
# Direct comparison table: annual R1 vs f0p residuals
annual_r1_stats = residual_stats(all_residuals[all_residuals['round_label']=='R1'], ['round_label','class_type'])
annual_r1_stats = annual_r1_stats.rename(columns={c: f'annual_R1_{c}' for c in annual_r1_stats.columns if c not in ['round_label','class_type']})

annual_r2_stats = residual_stats(all_residuals[all_residuals['round_label']=='R2'], ['round_label','class_type'])
annual_r2_stats = annual_r2_stats.rename(columns={c: f'annual_R2_{c}' for c in annual_r2_stats.columns if c not in ['round_label','class_type']})

annual_r3_stats = residual_stats(all_residuals[all_residuals['round_label']=='R3'], ['round_label','class_type'])
annual_r3_stats = annual_r3_stats.rename(columns={c: f'annual_R3_{c}' for c in annual_r3_stats.columns if c not in ['round_label','class_type']})

f0p_comp = f0p_stats.rename(columns={c: f'f0p_{c}' for c in f0p_stats.columns if c != 'class_type'})

print('\n=== Comparison: Annual R1 vs R2 vs R3 vs f0p ===')
print('\n--- Annual R1 ---')
print(annual_r1_stats.to_string(index=False))
print('\n--- Annual R2 ---')
print(annual_r2_stats.to_string(index=False))
print('\n--- Annual R3 ---')
print(annual_r3_stats.to_string(index=False))
print('\n--- f0p ---')
print(f0p_comp.to_string(index=False))

## 2.4 aq4 Special Analysis

In [ ]:
# Compare aq4 vs aq1-aq3 residual stats
r1_residuals = all_residuals[all_residuals['round_label'] == 'R1'].copy()
r1_residuals['is_aq4'] = r1_residuals['period_type'] == 'aq4'

aq4_comp = r1_residuals.groupby(['is_aq4', 'class_type']).agg(
    n=('residual', 'size'),
    mean_abs_residual=('abs_residual', 'mean'),
    median_abs_residual=('abs_residual', 'median'),
    p90_abs=('abs_residual', lambda x: x.quantile(0.90)),
    p95_abs=('abs_residual', lambda x: x.quantile(0.95)),
    mean_residual=('residual', 'mean'),
    direction_accuracy=('sign_match', 'mean'),
).reset_index()

print('=== aq4 vs aq1-aq3 (R1 only) ===')
print(aq4_comp.to_string(index=False))

# Per-quarter breakdown for R1
per_q = residual_stats(r1_residuals, ['period_type', 'class_type'])
print('\n=== R1 Residual Stats by Quarter x Class ===')
print(per_q.to_string(index=False))

## 2.5 R2/R3 Training Data Feasibility

In [ ]:
# Count R2/R3 data by |baseline| bins (matching f2p bin thresholds)
r23_data = all_residuals[all_residuals['round_label'].isin(['R2','R3'])].copy()
r23_data['abs_baseline'] = r23_data['baseline'].abs()

bins = [0, 50, 250, 1000, float('inf')]
labels = ['tiny(<50)', 'small(50-250)', 'medium(250-1000)', 'large(1000+)']
r23_data['mtm_bin'] = pd.cut(r23_data['abs_baseline'], bins=bins, labels=labels, right=False)

# Per PY x round x bin
bin_counts = r23_data.groupby(['planning_year', 'round_label', 'class_type', 'mtm_bin']).agg(
    count=('residual', 'size'),
).reset_index()
print('=== R2/R3 Bin Counts by PY x Round x Class ===')
print(bin_counts.to_string(index=False))

# Summary across PYs
bin_summary = r23_data.groupby(['round_label', 'class_type', 'mtm_bin']).agg(
    total_count=('residual', 'size'),
    n_pys=('planning_year', 'nunique'),
    avg_per_py=('residual', lambda x: len(x) / x.index.to_frame()['planning_year'].nunique() if hasattr(x.index, 'to_frame') else len(x)),
).reset_index()
# Recompute avg_per_py properly
bin_summary['avg_per_py'] = bin_summary['total_count'] / bin_summary['n_pys']
print('\n=== R2/R3 Bin Summary (all PYs) ===')
print(bin_summary.to_string(index=False))
print(f'\nf2p MIN_ROWS_FOR_EMPIRICAL = 100')
print(f'Bins with avg_per_py >= 100: {(bin_summary["avg_per_py"] >= 100).sum()} / {len(bin_summary)}')

In [ ]:
# If we pool across quarters and PYs for R2/R3, how many rows per bin?
pooled = r23_data.groupby(['round_label', 'class_type', 'mtm_bin']).agg(
    total_rows=('residual', 'size'),
    mean_abs_residual=('abs_residual', 'mean'),
    p90_abs=('abs_residual', lambda x: x.quantile(0.90)),
    p95_abs=('abs_residual', lambda x: x.quantile(0.95)),
).reset_index()

print('=== R2/R3 Pooled (all PYs, all quarters) ===')
print(pooled.to_string(index=False))
print(f'\nTotal R2/R3 rows: {len(r23_data)}')